# Neural Machine Translation — English → Arabic
### GRU Encoder + Bahdanau Attention + GRU Decoder

**Architecture overview:**
```
[EN sentence]
      ↓  embed
 Bidirectional GRU Encoder  →  h₁ … hₙ
      ↓
 Bahdanau Attention          →  score(sₜ,hⱼ) = vᵀ tanh(W₁sₜ + W₂hⱼ)
      ↓  context  cₜ = Σ αⱼhⱼ
 GRU Decoder  [embed(y) ; cₜ] → sₜ → FC → logits
      ↓  greedy / beam-search
[AR sentence]
```
**Reference:** Bahdanau et al., 2015 — *Neural Machine Translation by Jointly Learning to Align and Translate*

## 0 · Install Dependencies

In [ ]:
!pip install sacrebleu --quiet
print('✅ Dependencies ready')

## 1 · Upload ara_.txt

In [ ]:
from google.colab import files

print('📂 Please upload ara_.txt when the dialog appears...')
uploaded = files.upload()

import os
# Find the uploaded file (handles any filename variation like 'ara_.txt' or 'ara .txt')
DATA_PATH = list(uploaded.keys())[0]
print(f'✅ File uploaded: "{DATA_PATH}"  ({os.path.getsize(DATA_PATH):,} bytes)')

# Quick sanity check — print first 5 lines
with open(DATA_PATH, encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f'  {line.strip()}')
        if i == 4: break

## 2 · Imports & Reproducibility

In [ ]:
import re, time, math, random, json, unicodedata, warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3 · Hyperparameters

In [ ]:
# ── Automatically use larger config on GPU, smaller on CPU ──────────────────
if DEVICE.type == 'cuda':
    MAX_LEN     = 20
    EMB_DIM     = 256
    HID_DIM     = 512
    BATCH_SIZE  = 64
    N_EPOCHS    = 30
    print('🚀 GPU config loaded')
else:
    MAX_LEN     = 15
    EMB_DIM     = 128
    HID_DIM     = 256
    BATCH_SIZE  = 128
    N_EPOCHS    = 20
    print('🐢 CPU config loaded (consider Runtime → Change runtime type → GPU)')

MIN_FREQ    = 2
ENC_DROPOUT = 0.3
DEC_DROPOUT = 0.3
CLIP        = 1.0
TF_RATIO    = 0.5
TRAIN_SPLIT = 0.9
LR          = 1e-3

PAD, SOS, EOS, UNK = '<pad>', '<sos>', '<eos>', '<unk>'
CKPT_PATH = '/content/best_nmt_model.pt'

print(f'Config → MAX_LEN={MAX_LEN}, EMB={EMB_DIM}, HID={HID_DIM}, '
      f'BATCH={BATCH_SIZE}, EPOCHS={N_EPOCHS}')

## 4 · Vocabulary

In [ ]:
class Vocabulary:
    """Bidirectional word ↔ index mapping with 4 special tokens."""

    def __init__(self, name: str):
        self.name = name
        self.w2i  = {PAD: 0, SOS: 1, EOS: 2, UNK: 3}
        self.i2w  = {v: k for k, v in self.w2i.items()}
        self.freq = Counter()

    def build(self, sentences, min_freq=MIN_FREQ):
        for sent in sentences:
            self.freq.update(sent)
        for word, cnt in self.freq.items():
            if cnt >= min_freq and word not in self.w2i:
                idx = len(self.w2i)
                self.w2i[word] = idx
                self.i2w[idx]  = word
        print(f'  [{self.name}] vocab = {len(self):,} tokens')

    def encode(self, tokens):
        return [self.w2i.get(t, self.w2i[UNK]) for t in tokens]

    def decode(self, indices, skip_special=True):
        skip = {self.w2i[PAD], self.w2i[SOS], self.w2i[EOS]}
        return [self.i2w.get(i, UNK) for i in indices
                if not (skip_special and i in skip)]

    def __len__(self): return len(self.w2i)

## 5 · Tokenisation & Data Loading

In [ ]:
def tokenize_en(text: str):
    text = text.lower().strip()
    text = re.sub(r'([.!?,])', r' \1 ', text)
    text = re.sub(r"[^a-z0-9?.!,' ]", ' ', text)
    return text.split()

def tokenize_ar(text: str):
    text = unicodedata.normalize('NFC', text.strip())
    text = re.sub(r'([?!.،,])', r' \1 ', text)
    return text.split()

def load_pairs(path: str, max_len: int = MAX_LEN):
    pairs = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 2:
                continue
            en_tok = tokenize_en(parts[0])
            ar_tok = tokenize_ar(parts[1])
            if 1 <= len(en_tok) <= max_len and 1 <= len(ar_tok) <= max_len:
                pairs.append((en_tok, ar_tok))
    print(f'Loaded {len(pairs):,} valid pairs  (max_len={max_len})')
    return pairs

# ── Load + split ─────────────────────────────────────────────────────────────
all_pairs   = load_pairs(DATA_PATH)
random.shuffle(all_pairs)
split       = int(len(all_pairs) * TRAIN_SPLIT)
train_pairs = all_pairs[:split]
val_pairs   = all_pairs[split:]
print(f'Train: {len(train_pairs):,}   Val: {len(val_pairs):,}')

print('\nSample pairs:')
for en, ar in random.sample(train_pairs, 4):
    print(f'  EN: {" ".join(en)}')
    print(f'  AR: {" ".join(ar)}\n')

In [ ]:
# ── Build vocabularies ────────────────────────────────────────────────────────
src_vocab = Vocabulary('EN')
tgt_vocab = Vocabulary('AR')
src_vocab.build([p[0] for p in train_pairs])
tgt_vocab.build([p[1] for p in train_pairs])

In [ ]:
# ── Sentence length distribution ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, idx, lang, color in zip(axes, [0, 1], ['EN', 'AR'], ['steelblue', 'tomato']):
    lengths = [len(p[idx]) for p in train_pairs]
    ax.hist(lengths, bins=range(1, MAX_LEN + 2), color=color, edgecolor='white', alpha=0.85)
    ax.axvline(np.mean(lengths), color='black', linestyle='--',
               label=f'Mean = {np.mean(lengths):.1f}')
    ax.set_title(f'{lang} Sentence Length Distribution', fontsize=13)
    ax.set_xlabel('Tokens'); ax.set_ylabel('Count')
    ax.legend()
plt.tight_layout()
plt.savefig('/content/length_dist.png', dpi=120)
plt.show()

## 6 · Dataset & DataLoader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, src_vocab, tgt_vocab):
        self.pairs     = pairs
        self.src_vocab = src_vocab
        self.tgt_vocab = tgt_vocab

    def __len__(self): return len(self.pairs)

    def __getitem__(self, idx):
        src_tok, tgt_tok = self.pairs[idx]
        src = torch.tensor(
            self.src_vocab.encode(src_tok) + [self.src_vocab.w2i[EOS]],
            dtype=torch.long)
        tgt = torch.tensor(
            [self.tgt_vocab.w2i[SOS]]
            + self.tgt_vocab.encode(tgt_tok)
            + [self.tgt_vocab.w2i[EOS]],
            dtype=torch.long)
        return src, tgt


def collate_fn(batch):
    src_b, tgt_b = zip(*batch)
    src_pad = nn.utils.rnn.pad_sequence(src_b, batch_first=False, padding_value=0)
    tgt_pad = nn.utils.rnn.pad_sequence(tgt_b, batch_first=False, padding_value=0)
    return src_pad, tgt_pad  # (T, B)


train_loader = DataLoader(
    TranslationDataset(train_pairs, src_vocab, tgt_vocab),
    batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_fn, num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'))

val_loader = DataLoader(
    TranslationDataset(val_pairs, src_vocab, tgt_vocab),
    batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2,
    pin_memory=(DEVICE.type == 'cuda'))

print(f'Train batches: {len(train_loader):,}   Val batches: {len(val_loader):,}')

## 7 · Model Architecture

In [ ]:
# ── 7a. Bidirectional GRU Encoder ────────────────────────────────────────────
class Encoder(nn.Module):
    """
    embed(src) → BiGRU → outputs (T, B, 2H)
                       → hidden  (B, H)  via tanh(W·[h_fwd ; h_bwd])
    """
    def __init__(self, vocab_size, emb_dim, hid_dim, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn       = nn.GRU(emb_dim, hid_dim, bidirectional=True, batch_first=False)
        self.fc        = nn.Linear(hid_dim * 2, hid_dim)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))        # (T, B, E)
        outputs, hidden = self.rnn(emb)                # (T,B,2H), (2,B,H)
        hidden = torch.tanh(
            self.fc(torch.cat([hidden[-2], hidden[-1]], dim=1)))  # (B, H)
        return outputs, hidden


# ── 7b. Bahdanau Attention ───────────────────────────────────────────────────
class BahdanauAttention(nn.Module):
    """
    energy = v^T · tanh( W₁·s  +  W₂·h )
    alpha  = softmax(energy)
    context = Σ alpha_j · h_j
    """
    def __init__(self, enc_hid, dec_hid):
        super().__init__()
        self.W1 = nn.Linear(dec_hid,     dec_hid, bias=False)
        self.W2 = nn.Linear(enc_hid * 2, dec_hid, bias=False)
        self.v  = nn.Linear(dec_hid, 1,            bias=False)

    def forward(self, s, h):
        # s: (B, H)   h: (T, B, 2H)
        T     = h.size(0)
        s_rep = s.unsqueeze(1).expand(-1, T, -1)         # (B, T, H)
        h_t   = h.permute(1, 0, 2)                       # (B, T, 2H)
        energy = self.v(torch.tanh(
            self.W1(s_rep) + self.W2(h_t))).squeeze(2)   # (B, T)
        alpha   = F.softmax(energy, dim=1)                # (B, T)
        context = torch.bmm(alpha.unsqueeze(1), h_t).squeeze(1)  # (B, 2H)
        return context, alpha


# ── 7c. GRU Decoder ──────────────────────────────────────────────────────────
class Decoder(nn.Module):
    """
    Each step t:
      context, α  = Attention(s_{t-1}, enc_outputs)
      rnn_input   = [ embed(y_{t-1}) ; context ]
      s_t         = GRU(rnn_input, s_{t-1})
      ŷ_t         = FC([ s_t ; context ; embed(y_{t-1}) ])
    """
    def __init__(self, vocab_size, emb_dim, enc_hid, dec_hid, dropout, attention):
        super().__init__()
        self.attn      = attention
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn       = nn.GRU(enc_hid * 2 + emb_dim, dec_hid, batch_first=False)
        self.fc_out    = nn.Linear(enc_hid * 2 + dec_hid + emb_dim, vocab_size)
        self.dropout   = nn.Dropout(dropout)

    def forward(self, y, s, enc_out):
        y      = y.unsqueeze(0)                              # (1, B)
        emb    = self.dropout(self.embedding(y))             # (1, B, E)
        ctx, α = self.attn(s, enc_out)                       # (B,2H), (B,T)
        rnn_in = torch.cat([emb, ctx.unsqueeze(0)], dim=2)   # (1, B, E+2H)
        out, h = self.rnn(rnn_in, s.unsqueeze(0))
        out = out.squeeze(0); h = h.squeeze(0); emb = emb.squeeze(0)
        pred = self.fc_out(torch.cat([out, ctx, emb], dim=1))  # (B, V)
        return pred, h, α


# ── 7d. Seq2Seq Wrapper ───────────────────────────────────────────────────────
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, tf_ratio=TF_RATIO):
        T, B    = tgt.shape
        V       = self.decoder.fc_out.out_features
        outputs = torch.zeros(T, B, V).to(self.device)
        enc_out, hidden = self.encoder(src)
        dec_in = tgt[0]  # <sos>
        for t in range(1, T):
            pred, hidden, _ = self.decoder(dec_in, hidden, enc_out)
            outputs[t] = pred
            dec_in = tgt[t] if random.random() < tf_ratio else pred.argmax(1)
        return outputs

    @torch.no_grad()
    def translate_greedy(self, src_tensor, sos_idx, eos_idx, max_len=60):
        self.eval()
        src = src_tensor.unsqueeze(1).to(self.device)
        enc_out, hidden = self.encoder(src)
        dec_in = torch.tensor([sos_idx], device=self.device)
        tokens, attn_weights = [], []
        for _ in range(max_len):
            pred, hidden, α = self.decoder(dec_in, hidden, enc_out)
            attn_weights.append(α.squeeze(0).cpu().numpy())
            top1 = pred.argmax(1)
            if top1.item() == eos_idx: break
            tokens.append(top1.item())
            dec_in = top1
        return tokens, np.array(attn_weights)

    @torch.no_grad()
    def translate_beam(self, src_tensor, sos_idx, eos_idx, beam_width=5, max_len=60):
        self.eval()
        src = src_tensor.unsqueeze(1).to(self.device)
        enc_out, hidden = self.encoder(src)
        beams = [(0.0, [], hidden)]
        for _ in range(max_len):
            all_cands = []
            for log_p, toks, h in beams:
                if toks and toks[-1] == eos_idx:
                    all_cands.append((log_p, toks, h)); continue
                dec_in = torch.tensor(
                    [toks[-1] if toks else sos_idx], device=self.device)
                pred, new_h, _ = self.decoder(dec_in, h, enc_out)
                lps, idx = F.log_softmax(pred, dim=1).squeeze(0).topk(beam_width)
                for lp, i in zip(lps.tolist(), idx.tolist()):
                    all_cands.append((log_p + lp, toks + [i], new_h))
            beams = sorted(all_cands, key=lambda x: x[0], reverse=True)[:beam_width]
            if all(t and t[-1] == eos_idx for _, t, _ in beams): break
        best = beams[0][1]
        return [t for t in best if t != eos_idx]

## 8 · Instantiate Model

In [ ]:
def init_weights(m):
    for name, p in m.named_parameters():
        nn.init.normal_(p.data, 0, 0.01) if 'weight' in name \
            else nn.init.constant_(p.data, 0)

attn    = BahdanauAttention(HID_DIM, HID_DIM)
encoder = Encoder(len(src_vocab), EMB_DIM, HID_DIM, ENC_DROPOUT)
decoder = Decoder(len(tgt_vocab), EMB_DIM, HID_DIM, HID_DIM, DEC_DROPOUT, attn)
model   = Seq2Seq(encoder, decoder, DEVICE).to(DEVICE)
model.apply(init_weights)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model ready  |  Trainable parameters: {n_params:,}')
print(model)

## 9 · Training Loop

In [ ]:
def train_epoch(model, loader, opt, crit, clip, tf):
    model.train()
    total = 0
    for src, tgt in loader:
        src, tgt = src.to(DEVICE), tgt.to(DEVICE)
        opt.zero_grad()
        out  = model(src, tgt, tf)
        loss = crit(out[1:].reshape(-1, out.shape[-1]), tgt[1:].reshape(-1))
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), clip)
        opt.step()
        total += loss.item()
    return total / len(loader)


def eval_epoch(model, loader, crit):
    model.eval()
    total = 0
    with torch.no_grad():
        for src, tgt in loader:
            src, tgt = src.to(DEVICE), tgt.to(DEVICE)
            out  = model(src, tgt, tf_ratio=0.0)
            loss = crit(out[1:].reshape(-1, out.shape[-1]), tgt[1:].reshape(-1))
            total += loss.item()
    return total / len(loader)

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=3, factor=0.5)
criterion = nn.CrossEntropyLoss(ignore_index=0)

history  = {'train': [], 'val': [], 'ppl': []}
best_val = float('inf')

print(f'Training {N_EPOCHS} epochs on {DEVICE}...\n')

for epoch in range(1, N_EPOCHS + 1):
    t0  = time.time()
    tr  = train_epoch(model, train_loader, optimizer, criterion, CLIP, TF_RATIO)
    val = eval_epoch(model, val_loader, criterion)
    scheduler.step(val)

    history['train'].append(tr)
    history['val'].append(val)
    history['ppl'].append(math.exp(val))

    tag = ' ← best' if val < best_val else ''
    if val < best_val:
        best_val = val
        torch.save(model.state_dict(), CKPT_PATH)

    print(f'Epoch {epoch:02d}/{N_EPOCHS} | '
          f'Train {tr:.4f} | Val {val:.4f} | '
          f'PPL {math.exp(val):.2f} | '
          f'{time.time()-t0:.0f}s{tag}')

print(f'\n✅ Done — best val loss: {best_val:.4f}  PPL: {math.exp(best_val):.2f}')

## 10 · Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train']) + 1)

axes[0].plot(epochs, history['train'], 'o-', label='Train Loss', color='steelblue')
axes[0].plot(epochs, history['val'],   's-', label='Val Loss',   color='tomato')
axes[0].set_title('Cross-Entropy Loss', fontsize=13)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, history['ppl'], 'D-', color='darkorange')
axes[1].set_title('Validation Perplexity', fontsize=13)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('PPL')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('/content/training_curves.png', dpi=150)
plt.show()

with open('/content/training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('History saved.')

## 11 · BLEU Evaluation

In [ ]:
def corpus_bleu(hypotheses, references):
    """BLEU-4 with add-1 smoothing. No external deps."""
    scores = []
    for hyp, ref in zip(hypotheses, references):
        if not hyp or not ref:
            scores.append(0.0); continue
        bp    = min(1.0, math.exp(1 - len(ref) / max(len(hyp), 1)))
        log_p = 0.0
        for n in range(1, 5):
            hng   = Counter(tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1))
            rng   = Counter(tuple(ref[i:i+n]) for i in range(len(ref)-n+1))
            clip  = sum(min(c, rng[g]) for g, c in hng.items())
            total = max(len(hyp) - n + 1, 0)
            log_p += math.log((clip + 1) / (total + 1))
        scores.append(bp * math.exp(log_p / 4))
    return sum(scores) / len(scores) * 100 if scores else 0.0


# ── Load best checkpoint ──────────────────────────────────────────────────────
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
sos_idx = tgt_vocab.w2i[SOS]
eos_idx = tgt_vocab.w2i[EOS]

# ── Compute BLEU ──────────────────────────────────────────────────────────────
N_EVAL = 300
sample = random.sample(val_pairs, min(N_EVAL, len(val_pairs)))
hyps_g, hyps_b, refs = [], [], []

for en_tok, ar_tok in sample:
    src_ids = src_vocab.encode(en_tok) + [src_vocab.w2i[EOS]]
    src_t   = torch.tensor(src_ids, dtype=torch.long)
    g_ids, _ = model.translate_greedy(src_t, sos_idx, eos_idx)
    b_ids    = model.translate_beam(src_t, sos_idx, eos_idx, beam_width=5)
    hyps_g.append(tgt_vocab.decode(g_ids))
    hyps_b.append(tgt_vocab.decode(b_ids))
    refs.append(ar_tok)

bleu_g = corpus_bleu(hyps_g, refs)
bleu_b = corpus_bleu(hyps_b, refs)
print(f'BLEU-4  Greedy  : {bleu_g:.2f}')
print(f'BLEU-4  Beam-5  : {bleu_b:.2f}')

# ── Breakdown by length ───────────────────────────────────────────────────────
print('\nBLEU-4 by source length:')
print(f'{"Bucket":<16} {"n":>5}  {"Greedy":>8}  {"Beam-5":>8}')
print('─' * 42)
for lo, hi, label in [(1,3,'1-3 tok'),(4,6,'4-6 tok'),(7,10,'7-10 tok'),(11,MAX_LEN,f'11-{MAX_LEN} tok')]:
    idxs = [i for i,(en,_) in enumerate(sample) if lo <= len(en) <= hi]
    if not idxs: continue
    g = corpus_bleu([hyps_g[i] for i in idxs], [refs[i] for i in idxs])
    b = corpus_bleu([hyps_b[i] for i in idxs], [refs[i] for i in idxs])
    print(f'{label:<16} {len(idxs):>5}  {g:>8.2f}  {b:>8.2f}')

## 12 · Attention Heatmaps

In [ ]:
def plot_attention(src_tokens, tgt_tokens, attn_matrix, title='', ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(6, len(src_tokens)),
                                        max(4, len(tgt_tokens))))
    im = ax.imshow(attn_matrix, cmap='viridis', aspect='auto', vmin=0, vmax=1)
    ax.set_xticks(range(len(src_tokens)))
    ax.set_yticks(range(len(tgt_tokens)))
    ax.set_xticklabels(src_tokens, rotation=45, ha='right', fontsize=10)
    ax.set_yticklabels(tgt_tokens, fontsize=10)
    ax.set_xlabel('Source (EN)', fontsize=11)
    ax.set_ylabel('Target (AR)', fontsize=11)
    ax.set_title(title, fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    return ax


def get_attention_data(sentence):
    en_tok  = tokenize_en(sentence)
    src_ids = src_vocab.encode(en_tok) + [src_vocab.w2i[EOS]]
    src_t   = torch.tensor(src_ids, dtype=torch.long)
    pred_ids, attn_mtx = model.translate_greedy(src_t, sos_idx, eos_idx)
    ar_tok  = tgt_vocab.decode(pred_ids)
    src_lbl = en_tok + ['<eos>']
    attn_mtx = np.array(attn_mtx)[:, :len(src_lbl)]
    return src_lbl, ar_tok, attn_mtx


demo_sentences = [
    'i know .',
    'she is happy .',
    'where are you going ?',
    'he ran to the station .',
]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, sent in zip(axes.flatten(), demo_sentences):
    src_lbl, tgt_lbl, attn = get_attention_data(sent)
    if not tgt_lbl:
        ax.set_visible(False); continue
    plot_attention(src_lbl, tgt_lbl, attn,
                   title=f'EN: "{sent}"\nAR: "{" ".join(tgt_lbl)}"', ax=ax)

plt.suptitle('Bahdanau Attention Heatmaps', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('/content/attention_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

## 13 · Interactive Translation Demo

In [ ]:
def translate(sentence: str, beam_width: int = 5, show_attention: bool = True):
    en_tok  = tokenize_en(sentence)
    src_ids = src_vocab.encode(en_tok) + [src_vocab.w2i[EOS]]
    src_t   = torch.tensor(src_ids, dtype=torch.long)

    g_ids, attn = model.translate_greedy(src_t, sos_idx, eos_idx)
    b_ids       = model.translate_beam(src_t, sos_idx, eos_idx, beam_width)

    print(f'Input    (EN) : {sentence}')
    print(f'Greedy   (AR) : {" ".join(tgt_vocab.decode(g_ids))}')
    print(f'Beam-{beam_width}  (AR) : {" ".join(tgt_vocab.decode(b_ids))}')

    if show_attention and g_ids:
        src_lbl  = en_tok + ['<eos>']
        tgt_lbl  = tgt_vocab.decode(g_ids)
        attn_m   = np.array(attn)[:, :len(src_lbl)]
        fig, ax  = plt.subplots(figsize=(max(6, len(src_lbl)+1),
                                         max(4, len(tgt_lbl)+1)))
        plot_attention(src_lbl, tgt_lbl, attn_m,
                       title=f'Attention: "{sentence}"', ax=ax)
        plt.tight_layout(); plt.show()


# ── Try your own sentences here ───────────────────────────────────────────────
test_sentences = [
    'i love you .',
    'hello , how are you ?',
    'the book is on the table .',
    'she does not know him .',
    'stop !',
]

for s in test_sentences:
    print('─' * 50)
    translate(s, show_attention=False)
print('─' * 50)

In [ ]:
# ── Single sentence with attention heatmap ────────────────────────────────────
translate('where are you going ?', beam_width=5, show_attention=True)

## 14 · Best & Worst Translations

In [ ]:
def sentence_bleu(hyp, ref):
    return corpus_bleu([hyp], [ref]) if hyp and ref else 0.0

scored = []
for en_tok, ar_tok in random.sample(val_pairs, min(200, len(val_pairs))):
    src_ids  = src_vocab.encode(en_tok) + [src_vocab.w2i[EOS]]
    src_t    = torch.tensor(src_ids, dtype=torch.long)
    b_ids    = model.translate_beam(src_t, sos_idx, eos_idx, beam_width=5)
    pred_tok = tgt_vocab.decode(b_ids)
    scored.append((sentence_bleu(pred_tok, ar_tok), en_tok, ar_tok, pred_tok))

scored.sort(key=lambda x: x[0])

print('10 WORST translations:\n')
for bleu_s, en, ref, pred in scored[:10]:
    print(f'  BLEU {bleu_s:.2f} | EN: {" ".join(en)}')
    print(f'         Ref : {" ".join(ref)}')
    print(f'         Pred: {" ".join(pred)}\n')

print('\n10 BEST translations:\n')
for bleu_s, en, ref, pred in scored[-10:][::-1]:
    print(f'  BLEU {bleu_s:.2f} | EN: {" ".join(en)}')
    print(f'         Ref : {" ".join(ref)}')
    print(f'         Pred: {" ".join(pred)}\n')

## 15 · Save Checkpoint & Download

In [ ]:
# ── Save full checkpoint (model + vocabs + history) ───────────────────────────
FULL_CKPT = '/content/nmt_full_checkpoint.pt'
torch.save({
    'model_state': model.state_dict(),
    'src_vocab':   src_vocab,
    'tgt_vocab':   tgt_vocab,
    'history':     history,
    'config': dict(
        MAX_LEN=MAX_LEN, EMB_DIM=EMB_DIM, HID_DIM=HID_DIM,
        ENC_DROPOUT=ENC_DROPOUT, DEC_DROPOUT=DEC_DROPOUT,
    )
}, FULL_CKPT)
print(f'Checkpoint saved to {FULL_CKPT}')

# ── Download to your local machine ───────────────────────────────────────────
from google.colab import files
files.download(FULL_CKPT)
files.download('/content/training_curves.png')
files.download('/content/attention_heatmaps.png')
print('✅ Files downloaded.')